# ED-Alert
### Ministerio de Educación – Tasa de Abandono

In [1]:
from pathlib import Path
import pandas as pd

#### Ministerio Tasa Abandono

In [2]:
path_df = Path("../data/rend_2_15.csv")
print(f"--- El csv rend_2_15 existe: {path_df.exists()}")

print("---Abrir archivo---")
with open(path_df, "r", encoding="utf-8-sig") as f:
    for i, line in enumerate(f):
        print(f"Línea {i}: {repr(line)}")
        if i > 5:
            break

df_min_ed = pd.read_csv(path_df,
                     encoding="utf-8-sig",
                     sep=";",
                     decimal=",",
                     na_values=[".", ".."])
print("\n---Dataset Ministerio cargado---")
print(df_min_ed.head())


df_pivot = df_min_ed[
    (df_min_ed["Comunidad autónoma"] != "TOTAL") &
    (df_min_ed["Sexo"] == "AMBOS SEXOS") &
    (df_min_ed["Situación educativa"] != "TOTAL")
].pivot_table(
    index=["Curso de ingreso", "Comunidad autónoma", "Años desde el ingreso", "Sexo"],
    columns="Situación educativa",
    values="Total",
    aggfunc="first"
).reset_index()

# Limpiar nombre del índice de columnas
df_pivot.columns.name = None

# Renombrar columnas para que sean más manejables
df_pivot = df_pivot.rename(columns={
    "Curso de ingreso": "curso",
    "Comunidad autónoma": "ccaa",
    "Años desde el ingreso": "años",
    "Sexo": "sexo",
    "Sin Matricula": "sin_matricula",
    "Matriculado mismo ciclo": "matriculado_mismo",
    "Matriculados otro ciclo": "matriculado_otro",
    "Bachillerato": "bachillerato",
    "Otras Enseñanzas": "otras_enseñanzas",
    "Terminan años anteriores": "terminan_antes"
})

print("\n--- Dataset Ministerio pivotado---")
print(df_pivot.head())

df_abandono = df_pivot[
    (df_pivot["curso"] == "2021-2022") &
    (df_pivot["años"] == "2º año")   # solo abandono temprano
][["ccaa", "sexo", "sin_matricula"]].copy()

print("\n---Dataset Ministerio filtrado---")
print(df_abandono.head())

"""sin_matricula — es tu proxy de abandono. El porcentaje de alumnos que después de X años 
ya no están matriculados en ningún sitio. Es lo más cercano a abandono real que tienes"""

--- El csv rend_2_15 existe: True
---Abrir archivo---
Línea 0: 'Curso de ingreso;Comunidad autónoma;Sexo;Años desde el ingreso;Situación educativa;Total\n'
Línea 1: '2021-2022;TOTAL;AMBOS SEXOS;2º año;TOTAL;100,0\n'
Línea 2: '2021-2022;TOTAL;AMBOS SEXOS;2º año;Terminan años anteriores;.\n'
Línea 3: '2021-2022;TOTAL;AMBOS SEXOS;2º año;Matriculado mismo ciclo;76,6\n'
Línea 4: '2021-2022;TOTAL;AMBOS SEXOS;2º año;Matriculados otro ciclo;6,0\n'
Línea 5: '2021-2022;TOTAL;AMBOS SEXOS;2º año;Bachillerato;0,9\n'
Línea 6: '2021-2022;TOTAL;AMBOS SEXOS;2º año;Otras Enseñanzas;0,7\n'

---Dataset Ministerio cargado---
  Curso de ingreso Comunidad autónoma         Sexo Años desde el ingreso  \
0        2021-2022              TOTAL  AMBOS SEXOS                2º año   
1        2021-2022              TOTAL  AMBOS SEXOS                2º año   
2        2021-2022              TOTAL  AMBOS SEXOS                2º año   
3        2021-2022              TOTAL  AMBOS SEXOS                2º año   
4       

'sin_matricula — es tu proxy de abandono. El porcentaje de alumnos que después de X años \nya no están matriculados en ningún sitio. Es lo más cercano a abandono real que tienes'

In [3]:
print(df_abandono.shape)
print(df_abandono.head(20))
print(df_abandono["ccaa"].unique())
print(df_abandono["sin_matricula"].describe())

(19, 3)
                            ccaa         sexo  sin_matricula
357                    Andalucía  AMBOS SEXOS           13.4
359                       Aragón  AMBOS SEXOS           16.3
361      Asturias, Principado de  AMBOS SEXOS           18.2
363               Balears, Illes  AMBOS SEXOS           22.2
365                     Canarias  AMBOS SEXOS           24.0
367                    Cantabria  AMBOS SEXOS           17.5
369              Castilla y León  AMBOS SEXOS           18.0
371           Castilla-La Mancha  AMBOS SEXOS           16.3
373                     Cataluña  AMBOS SEXOS           12.4
375                        Ceuta  AMBOS SEXOS           23.5
377         Comunitat Valenciana  AMBOS SEXOS           16.6
379                  Extremadura  AMBOS SEXOS           13.4
381                      Galicia  AMBOS SEXOS           18.5
383         Madrid, Comunidad de  AMBOS SEXOS           14.2
385                      Melilla  AMBOS SEXOS           19.9
387           Mu

In [4]:
nombre_map = {
    "Asturias, Principado de"      : "Asturias",
    "Balears, Illes"               : "Islas Baleares",
    "Madrid, Comunidad de"         : "Madrid",
    "Murcia (Región de)"           : "Murcia",
    "Navarra, Comunidad Foral de"  : "Navarra",
    "Rioja, La"                    : "La Rioja",
    "Comunitat Valenciana"         : "Comunidad Valenciana"
}

df_abandono["ccaa"] = df_abandono["ccaa"].replace(nombre_map)

# Exportar
df_abandono.to_csv("../data/02_ministerio_abandono_fp_ccaa.csv", index=False)

Las visualizaciones más adecuadas para estos datos:
1. Mapa choropleth de España — la más impactante
Cada CCAA coloreada por tasa de abandono. El usuario ve de un vistazo dónde es más grave.
En Tableau conectas el CSV y usas el campo geográfico de CCAA. Necesitarás estandarizar los nombres para que Tableau los reconozca.
2. Gráfico de barras horizontal ordenado — el más legible
CCAAas ordenadas de mayor a menor abandono, con una línea vertical en la media nacional (17.1%).
3. KPI de portada
El dato más impactante como número grande: "1 de cada 6 alumnos de FP Grado Medio abandona antes del segundo año"